In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from scipy.interpolate import griddata
from tqdm import tqdm
from pyproj import CRS
from scipy.ndimage import distance_transform_edt

In [3]:
soil_propeties = ["pH", "CE", "CEC", "OM", "A%", "L%", "S%"]
soil_propeties = ["CEC"]

model = "LGBM"
model_abrev= 'lgbm'
for soil_prop in soil_propeties:
    df = pd.read_csv(f"./predictions/{model}/{model_abrev}_predictions_{soil_prop}.csv")
    print(df.info())
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.X, df.Y), crs="EPSG:4326")
    gdf = gdf.to_crs(epsg=32631)  # UTM zone 31N

    df['X_m'] = gdf.geometry.x
    df['Y_m'] = gdf.geometry.y

    # Use projected coordinates (UTM)
    x = df['X_m'].values
    y = df['Y_m'].values
    values = df[soil_prop].values

    pixel_size = 1000  # meters

    # Grid limits
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    # Grid size
    nx = int(np.ceil((xmax - xmin) / pixel_size))
    ny = int(np.ceil((ymax - ymin) / pixel_size))

    # Convert points to grid indices
    ix = ((x - xmin) / pixel_size).astype(int)
    iy = ((ymax - y) / pixel_size).astype(int)  # flip y

    # Initialize raster with NaNs
    grid = np.full((ny, nx), np.nan, dtype=np.float32)

    # Assign values (nearest)
    grid[iy, ix] = values

    # Fill NaNs using nearest neighbor
    from scipy.ndimage import distance_transform_edt

    mask = np.isnan(grid)
    coords = np.array(np.nonzero(~mask)).T
    values_known = grid[~mask]

    # Compute distance transform to nearest known pixel
    filled_grid = distance_transform_edt(mask, return_distances=False, return_indices=True)
    grid_filled = grid[tuple(filled_grid)]

    # Save GeoTIFF
    transform = from_origin(xmin, ymax, pixel_size, pixel_size)

    with rasterio.open(
        f'./maps/{model}/{model}_{soil_prop}_map.tif',
        'w',
        driver='GTiff',
        height=grid_filled.shape[0],
        width=grid_filled.shape[1],
        count=1,
        dtype=grid_filled.dtype,
        crs='EPSG:32631',
        transform=transform
    ) as dst:
        dst.write(grid_filled, 1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823300 entries, 0 to 2823299
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   Unnamed: 0  int64  
 1   X           float64
 2   Y           float64
 3   CEC         float64
dtypes: float64(3), int64(1)
memory usage: 86.2 MB
None
